# Competitor Evaluation Dashboard

This notebook provides a detailed performance analysis of different Approximate Nearest Neighbor (ANN) search algorithms evaluated under the **Orthogonal School** competition harness. It connects to the SQLite database `orthogonal-competition/results.db` to extract, run, and visualize benchmarking results for each competitor.

---

## 0. Database Management & Running Evaluations

Use the cells below to manage your database (e.g., clearing past results) and running new evaluations directly from this notebook.

In [ ]:
# === CLEAR / RESET BENCHMARK DATABASE ===
# Uncomment and run the code below to delete all runs from the database and start fresh.

# import sqlite3
# conn = sqlite3.connect('./orthogonal-competition/results.db')
# conn.execute("DROP TABLE IF EXISTS runs")
# conn.execute("DROP TABLE IF EXISTS detail")
# conn.commit()
# conn.close()
# print("Database cleared successfully!")

In [ ]:
# === RUN EVALUATION FOR A SPECIFIC TESTCASE ===
# Modify the parameters below and run this cell to benchmark a competitor on a dataset.

TEAM_NAME = "faiss-hnsw"
DOCKER_IMAGE = "ann-orthogonal/faiss-hnsw:latest"
DATASET_PATH = "./dataset/yahoo-minilm-public.hdf5"  # Path to dataset HDF5 file

!python3 orthogonal-competition/evaluator.py evaluate \
    --team {TEAM_NAME} \
    --image {DOCKER_IMAGE} \
    --dataset {DATASET_PATH} \
    --db ./orthogonal-competition/results.db

## 1. Load Data & Parse Metrics

We load successful runs from `results.db` and parse the detailed query latency parameters from `extra_metrics` JSON blob.

In [ ]:
import sqlite3
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

db_path = './orthogonal-competition/results.db'
if not os.path.exists(db_path):
    print(f"Database not found at {db_path}. Running an evaluation above will initialize it.")
    df = pd.DataFrame()
else:
    conn = sqlite3.connect(db_path)
    try:
        df = pd.read_sql_query("SELECT * FROM runs WHERE status='success'", conn)
    except Exception as e:
        print("No results tables found or database is empty.")
        df = pd.DataFrame()
    conn.close()

if not df.empty:
    # Helper functions to extract latency metrics from extra_metrics JSON
    def get_latency_metric(extra_str, metric):
        try:
            meta = json.loads(extra_str)
            return meta.get("latency_ms", {}).get(metric, np.nan)
        except:
            return np.nan

    df['mean_latency_ms'] = df['extra_metrics'].apply(lambda x: get_latency_metric(x, 'mean'))
    df['median_latency_ms'] = df['extra_metrics'].apply(lambda x: get_latency_metric(x, 'median'))
    df['p95_latency_ms'] = df['extra_metrics'].apply(lambda x: get_latency_metric(x, 'p95'))
    df['p99_latency_ms'] = df['extra_metrics'].apply(lambda x: get_latency_metric(x, 'p99'))
    print(f"Loaded {len(df)} successful benchmark runs.")
else:
    print("No successful benchmark runs available. Please run an evaluation using the cell above first.")

## 2. Raw Specification Benchmark Results

Below is the complete overview table of raw metrics for each competitor, grouped by dataset and scenario specification.

In [ ]:
if not df.empty:
    cols = ['dataset', 'scenario', 'team_name', 'avg_recall', 'qps', 'build_time_s', 'peak_mem_mb', 'index_mem_mb', 'n_dist_queries', 'mean_latency_ms', 'p95_latency_ms']
    df_summary = df[cols].sort_values(by=['dataset', 'scenario', 'team_name'])
    display(df_summary.style.format({
        "avg_recall": "{:.4f}",
        "qps": "{:,.1f}",
        "build_time_s": "{:.4f}s",
        "peak_mem_mb": "{:.1f} MB",
        "index_mem_mb": "{:.1f} MB",
        "n_dist_queries": "{:,}",
        "mean_latency_ms": "{:.3f} ms",
        "p95_latency_ms": "{:.3f} ms"
    }).set_caption("Raw Competitor Metrics per Dataset and Scenario"))
else:
    print("No data available to display.")

## 3. Speed vs. Accuracy Trade-Off (QPS vs. Recall)

The core trade-off in Approximate Nearest Neighbor (ANN) search is speed (measured in Queries Per Second, QPS) vs. accuracy (measured in Average Recall). 
- Algorithms located in the **top-right** quadrant are the best, achieving high accuracy with minimal query latency.
- Scatter plots are generated below for each distinct dataset.

In [ ]:
if not df.empty:
    datasets = df['dataset'].unique()
    for ds in datasets:
        df_ds = df[df['dataset'] == ds]
        plt.figure(figsize=(10, 6))
        sns.scatterplot(data=df_ds, x="avg_recall", y="qps", hue="team_name", style="scenario", s=180, alpha=0.9)
        plt.title(f"QPS vs. Average Recall on Dataset: '{ds}'\n(Top-Right is best: High QPS, High Recall)", fontsize=14, fontweight='bold')
        plt.xlabel("Average Recall")
        plt.ylabel("Queries Per Second (QPS) - Log Scale")
        plt.yscale("log")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
else:
    print("No data available to display.")

## 4. Index Build Time & Index Memory Overhead

Index building time (`build_time_s`) is critical for fast startup or dynamic indices. Index memory footprint (`index_mem_mb`) measures how much RAM is required by the index itself (over the raw database vectors).

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: Index Build Time comparison
    sns.barplot(data=df, x="scenario", y="build_time_s", hue="team_name", ax=axes[0], palette="Set2")
    axes[0].set_title("Index Build Time (Seconds)\n(Lower is better)", fontsize=13, fontweight='bold')
    axes[0].set_ylabel("Build Time (s) - Log Scale")
    axes[0].set_yscale("log")

    # Right: Index Memory footprint
    sns.barplot(data=df, x="scenario", y="index_mem_mb", hue="team_name", ax=axes[1], palette="Set2")
    axes[1].set_title("Index Memory Footprint (MB)\n(Lower is better)", fontsize=13, fontweight='bold')
    axes[1].set_ylabel("Memory (MB)")

    plt.tight_layout()
    plt.show()
else:
    print("No data available to display.")

## 5. Distance Computations Count (`n_dist_queries`)

This metric counts the total number of full vector distance computations performed during query evaluation. An algorithm that prunes the search space effectively will compute far fewer distances.

In [ ]:
if not df.empty:
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df, x="scenario", y="n_dist_queries", hue="team_name", palette="coolwarm")
    plt.title("Total Distance Computations (Log Scale)\n(Lower is better)", fontsize=14, fontweight='bold')
    plt.ylabel("Distance Calculations (Count)")
    plt.yscale("log")
    plt.tight_layout()
    plt.show()
else:
    print("No data available to display.")

## 6. Latency Profile (Average vs. P95 Tail Latency)

While QPS measures total system throughput, individual query response times (latency) are important for user-facing applications. We compare the average (mean) query latency against the P95 tail latency (the 95th percentile worst-case latency).

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: Mean Latency
    sns.barplot(data=df, x="scenario", y="mean_latency_ms", hue="team_name", ax=axes[0], palette="muted")
    axes[0].set_title("Mean Query Latency (ms)\n(Lower is better)", fontsize=13, fontweight='bold')
    axes[0].set_ylabel("Mean Latency (ms)")

    # Right: P95 Tail Latency
    sns.barplot(data=df, x="scenario", y="p95_latency_ms", hue="team_name", ax=axes[1], palette="muted")
    axes[1].set_title("P95 Tail Latency (ms)\n(Lower is better)", fontsize=13, fontweight='bold')
    axes[1].set_ylabel("P95 Latency (ms)")

    plt.tight_layout()
    plt.show()
else:
    print("No data available to display.")

## 7. Competitor Performance Analysis Summary

Based on the empirical measurements, here is a performance breakdown of each competitor:

1. **faiss-hnsw**:
   - **Strengths**: Achieves high QPS and excellent recall-speed trade-offs on most datasets.
   - **Weaknesses**: Has longer index build times (`build_time_s`) and high memory footprint (`index_mem_mb`).

2. **LSH_Cpp**:
   - **Strengths**: Has extremely fast build times. Prunes distance queries well on fast/ultra-fast parameters.
   - **Weaknesses**: Suffers in recall-speed trade-offs when high accuracy (>= 0.95 recall) is required, needing many hash tables that increase memory usage.

3. **ANNvedi**:
   - **Strengths**: Our implementation is evaluated across scenarios (`high_recall`, `fast`, `memory`, `lsh_default`, etc.). Compare its raw QPS, recall, and distance query counts to find optimal performance configuration trade-offs.
   - **Action**: Check if your model parameters (like backend selection, construction limits, quantization) match the efficiency levels of `faiss-hnsw`.